# Zero-shot NLI baseline

Generated by `scripts/build_kaggle_notebook.py`, so edit the repo instead of this notebook.

**Before running:** in the right panel, set *Accelerator* to **GPU T4** and turn **Internet** on
(Internet needs a phone-verified Kaggle account). Then use *Run All*. When it finishes, download
`results.zip` from the *Output* section and unzip it into the repo's `results/` folder.

In [ ]:
!mkdir -p data src/hd eval results

In [ ]:
%%writefile data/prepare.py
"""RAGTruth -> sentence-level pairs.

Reads data/raw/{response,source_info}.jsonl (from github.com/ParticleMedia/RAGTruth) and writes data/processed/:
  sources.jsonl              one row per source: source_id, task_type, text (stored once, not per sentence)
  {train,val,test}.jsonl     one row per response sentence: source_id, sentence, label (1 = hallucinated), ...
  stats.json                 class balance and source lengths per split

Validation is carved out of train by source_id, because every source has responses from 6 LLMs and a random
split would leak. Standard library only, so it runs anywhere (laptop or Kaggle) in a few seconds.
"""

import argparse
import hashlib
import json
import re
import statistics
from collections import Counter
from pathlib import Path

ROOT = Path(__file__).resolve().parent
# A sentence ends at . ! or ? (optionally followed by a closing quote or bracket) and whitespace, or at a newline.
SENTENCE_END = re.compile(r"(?<=[.!?])[\"')\]]?\s+|\n+")
# Initials, common abbreviations and list numbers shouldn't end a sentence ("Dr. Smith", "J. K. Rowling", "1. Open").
NO_SPLIT_BEFORE = re.compile(r"(\b[A-Z]|\b(?:Mr|Mrs|Ms|Dr|St|Jr|Sr|vs|etc|e\.g|i\.e|No|Inc|Ltd|Co|U\.S)|^\s*\d{1,2})\.$")


def split_sentences(text: str) -> list[tuple[int, int]]:
    """Return (start, end) character offsets of each sentence, so annotated spans can be mapped onto them."""
    spans, start = [], 0
    for m in SENTENCE_END.finditer(text):
        if m.group().strip() == "" and NO_SPLIT_BEFORE.search(text[start : m.start()]):
            continue
        spans.append((start, m.start() + len(m.group().rstrip())))
        start = m.end()
    spans.append((start, len(text)))
    # Trim surrounding whitespace and drop empty pieces.
    out = []
    for s, e in spans:
        piece = text[s:e]
        if piece.strip():
            lead = len(piece) - len(piece.lstrip())
            out.append((s + lead, s + len(piece.rstrip())))
    return out


def source_text(src: dict) -> str:
    info = src["source_info"]
    if src["task_type"] == "QA":
        return f"Question: {info['question']}\n\n{info['passages']}"
    if src["task_type"] == "Data2txt":
        return json.dumps(info, ensure_ascii=False)
    return info  # Summary: the article itself


def split_of(row: dict, val_frac: float) -> str:
    if row["split"] == "test":
        return "test"
    bucket = int(hashlib.sha256(row["source_id"].encode()).hexdigest(), 16) % 1000
    return "val" if bucket < val_frac * 1000 else "train"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", type=Path, default=ROOT / "raw")
    ap.add_argument("--out", type=Path, default=ROOT / "processed")
    ap.add_argument("--val-frac", type=float, default=0.1)
    args = ap.parse_args()
    args.out.mkdir(parents=True, exist_ok=True)

    sources = {}
    with open(args.raw / "source_info.jsonl") as f:
        for line in f:
            s = json.loads(line)
            sources[s["source_id"]] = {"source_id": s["source_id"], "task_type": s["task_type"], "text": source_text(s)}
    with open(args.out / "sources.jsonl", "w") as f:
        for s in sources.values():
            f.write(json.dumps(s, ensure_ascii=False) + "\n")

    rows = {"train": [], "val": [], "test": []}
    skipped = Counter()
    with open(args.raw / "response.jsonl") as f:
        for line in f:
            r = json.loads(line)
            if r["quality"] != "good":  # incorrect refusals and truncated responses
                skipped[r["quality"]] += 1
                continue
            split = split_of(r, args.val_frac)
            for i, (s, e) in enumerate(split_sentences(r["response"])):
                overlapping = [lab for lab in r["labels"] if lab["start"] < e and lab["end"] > s]
                rows[split].append({
                    "id": f"{r['id']}-{i}",
                    "response_id": r["id"],
                    "source_id": r["source_id"],
                    "task_type": sources[r["source_id"]]["task_type"],
                    "model": r["model"],
                    "sentence_idx": i,
                    "sentence": r["response"][s:e],
                    "label": int(bool(overlapping)),
                    "label_types": sorted({lab["label_type"] for lab in overlapping}),
                })

    ids = {k: {row["source_id"] for row in v} for k, v in rows.items()}
    assert not ids["train"] & ids["val"] and not ids["train"] & ids["test"] and not ids["val"] & ids["test"], \
        "source_id leaks across splits"

    stats = {"skipped_responses": dict(skipped)}
    for split, items in rows.items():
        with open(args.out / f"{split}.jsonl", "w") as f:
            for row in items:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        src_words = [len(sources[sid]["text"].split()) for sid in ids[split]]
        pos = sum(row["label"] for row in items)
        stats[split] = {
            "sentences": len(items),
            "responses": len({row["response_id"] for row in items}),
            "sources": len(ids[split]),
            "hallucinated": pos,
            "hallucinated_frac": round(pos / len(items), 4),
            "hallucinated_frac_by_task": {
                t: round(sum(x["label"] for x in items if x["task_type"] == t)
                         / max(1, sum(x["task_type"] == t for x in items)), 4)
                for t in ("QA", "Summary", "Data2txt")
            },
            "source_words": {"median": statistics.median(src_words), "p95": sorted(src_words)[int(0.95 * len(src_words))],
                             "max": max(src_words)},
            "sentence_words_median": statistics.median(len(x["sentence"].split()) for x in items),
        }
    (args.out / "stats.json").write_text(json.dumps(stats, indent=2) + "\n")
    print(json.dumps(stats, indent=2))
    print("No source_id appears in more than one split.")


if __name__ == "__main__":
    main()

In [ ]:
!touch src/hd/__init__.py

In [ ]:
%%writefile src/hd/data.py
"""Load the processed RAGTruth files written by data/prepare.py."""

import json
from pathlib import Path

DATA = Path(__file__).resolve().parents[2] / "data" / "processed"


def read_jsonl(path: Path) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]


def load_sources(data_dir: Path = DATA) -> dict[str, str]:
    return {s["source_id"]: s["text"] for s in read_jsonl(Path(data_dir) / "sources.jsonl")}


def load_split(split: str, data_dir: Path = DATA, limit: int | None = None) -> list[dict]:
    rows = read_jsonl(Path(data_dir) / f"{split}.jsonl")
    return rows[:limit] if limit else rows


def chunk_words(text: str, size: int = 250, overlap: int | None = None) -> list[str]:
    """Split a long source into overlapping word windows that fit a 512-token encoder alongside the sentence."""
    overlap = size // 5 if overlap is None else overlap
    if not 0 <= overlap < size:
        raise ValueError("overlap must be in [0, size)")
    words = text.split()
    if len(words) <= size:
        return [" ".join(words)]
    step = size - overlap
    return [" ".join(words[i : i + size]) for i in range(0, len(words) - overlap, step)]

In [ ]:
%%writefile src/hd/metrics.py
"""Metrics for the hallucinated class (label 1). Plain Python, so they run and test anywhere."""


def prf(labels: list[int], preds: list[int]) -> dict:
    tp = sum(1 for y, p in zip(labels, preds) if y and p)
    fp = sum(1 for y, p in zip(labels, preds) if not y and p)
    fn = sum(1 for y, p in zip(labels, preds) if y and not p)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


def best_threshold(labels: list[int], scores: list[float]) -> float:
    """Threshold t maximizing F1 when predicting hallucinated for score >= t."""
    pairs = sorted(zip(scores, labels), reverse=True)
    positives = sum(labels)
    tp = fp = 0
    best_f1, best_t = -1.0, 1.0
    for i, (s, y) in enumerate(pairs):
        tp += y
        fp += 1 - y
        if i + 1 < len(pairs) and pairs[i + 1][0] == s:
            continue  # only cut between distinct scores
        f1 = 2 * tp / (tp + fp + positives) if positives else 0.0
        if f1 > best_f1:
            best_f1, best_t = f1, s
    return best_t


def auroc(labels: list[int], scores: list[float]) -> float:
    """Area under the ROC curve via the Mann-Whitney rank statistic, with tied scores sharing an average rank."""
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    ranks = [0.0] * len(scores)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        for k in range(i, j + 1):
            ranks[order[k]] = (i + j) / 2 + 1
        i = j + 1
    n_pos = sum(labels)
    n_neg = len(labels) - n_pos
    if not n_pos or not n_neg:
        return float("nan")
    rank_sum = sum(r for r, y in zip(ranks, labels) if y)
    return (rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def report(labels: list[int], scores: list[float], threshold: float) -> dict:
    out = prf(labels, [int(s >= threshold) for s in scores])
    out["auroc"] = auroc(labels, scores)
    out = {k: round(v, 4) for k, v in out.items()}
    out.update(n=len(labels), hallucinated=sum(labels))
    return out

In [ ]:
%%writefile src/hd/baselines.py
"""Zero-shot NLI baseline: a sentence is hallucinated when no chunk of its source entails it.

score = 1 - max over source chunks of P(entailment | chunk, sentence), so higher means more likely hallucinated.
No training. Run on a GPU (Kaggle): PYTHONPATH=src python -m hd.baselines --splits val test
"""

import argparse
import json
import time
from pathlib import Path

from hd.data import chunk_words, load_sources, load_split

NLI_MODEL = "cross-encoder/nli-deberta-v3-base"


class NLIScorer:
    def __init__(self, model_name: str = NLI_MODEL, batch_size: int = 64):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.torch = torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device).eval()
        if self.device == "cuda":
            self.model.half()
        labels = {name.lower(): i for i, name in self.model.config.id2label.items()}
        self.entail_idx = labels["entailment"]
        self.batch_size = batch_size

    def __call__(self, pairs: list[tuple[str, str]]) -> list[float]:
        """P(entailment) for each (premise, hypothesis) pair. Pairs are batched by length to cut padding."""
        order = sorted(range(len(pairs)), key=lambda i: len(pairs[i][0]))
        probs = [0.0] * len(pairs)
        with self.torch.inference_mode():
            for b in range(0, len(order), self.batch_size):
                idx = order[b : b + self.batch_size]
                enc = self.tok([pairs[i][0] for i in idx], [pairs[i][1] for i in idx], truncation="only_first",
                               max_length=512, padding=True, return_tensors="pt").to(self.device)
                p = self.model(**enc).logits.float().softmax(-1)[:, self.entail_idx].tolist()
                for i, v in zip(idx, p):
                    probs[i] = v
        return probs


def score_rows(rows: list[dict], sources: dict[str, str], entailment, chunk_size: int = 250) -> tuple[list[float], int]:
    """Return (hallucination score per row, number of NLI pairs run)."""
    chunks = {sid: chunk_words(sources[sid], chunk_size) for sid in {r["source_id"] for r in rows}}
    pairs, owner = [], []
    for i, r in enumerate(rows):
        for c in chunks[r["source_id"]]:
            pairs.append((c, r["sentence"]))
            owner.append(i)
    best = [0.0] * len(rows)
    for i, p in zip(owner, entailment(pairs)):
        best[i] = max(best[i], p)
    return [1.0 - b for b in best], len(pairs)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--splits", nargs="+", default=["val", "test"])
    ap.add_argument("--limit", type=int, help="score only the first N sentences per split (smoke run)")
    ap.add_argument("--out", type=Path, default=Path("results"))
    args = ap.parse_args()
    args.out.mkdir(parents=True, exist_ok=True)

    sources = load_sources()
    scorer = NLIScorer()
    gpu = scorer.torch.cuda.get_device_name(0) if scorer.device == "cuda" else "cpu"
    timing = {"model": NLI_MODEL, "device": gpu}
    for split in args.splits:
        rows = load_split(split, limit=args.limit)
        start = time.perf_counter()
        scores, n_pairs = score_rows(rows, sources, scorer)
        seconds = time.perf_counter() - start
        with open(args.out / f"nli_{split}.jsonl", "w") as f:
            for r, s in zip(rows, scores):
                f.write(json.dumps({"id": r["id"], "task_type": r["task_type"], "model": r["model"],
                                    "label": r["label"], "score": round(s, 6)}) + "\n")
        timing[split] = {"sentences": len(rows), "nli_pairs": n_pairs, "seconds": round(seconds, 1),
                         "ms_per_sentence": round(1000 * seconds / len(rows), 2)}
        print(split, timing[split], flush=True)
    (args.out / "nli_timing.json").write_text(json.dumps(timing, indent=2) + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile eval/evaluate.py
"""Metrics from saved predictions, so every reported number can be reproduced from files in results/.

The threshold is picked on validation F1 and then applied unchanged to test.
Usage: python eval/evaluate.py --method nli   (reads results/nli_{val,test}.jsonl, writes results/nli_metrics.json)
"""

import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from hd.data import read_jsonl  # noqa: E402
from hd.metrics import best_threshold, report  # noqa: E402


def grouped(rows: list[dict], key: str, threshold: float) -> dict:
    out = {}
    for value in sorted({r[key] for r in rows}):
        sub = [r for r in rows if r[key] == value]
        out[value] = report([r["label"] for r in sub], [r["score"] for r in sub], threshold)
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True)
    ap.add_argument("--results", type=Path, default=Path("results"))
    args = ap.parse_args()

    val = read_jsonl(args.results / f"{args.method}_val.jsonl")
    test = read_jsonl(args.results / f"{args.method}_test.jsonl")
    t = best_threshold([r["label"] for r in val], [r["score"] for r in val])
    metrics = {
        "method": args.method,
        "threshold_from_val": round(t, 6),
        "val": report([r["label"] for r in val], [r["score"] for r in val], t),
        "test": report([r["label"] for r in test], [r["score"] for r in test], t),
        "test_by_task": grouped(test, "task_type", t),
        "test_by_llm": grouped(test, "model", t),
    }
    (args.results / f"{args.method}_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
!mkdir -p data/raw
!curl -sfL -o data/raw/response.jsonl https://raw.githubusercontent.com/ParticleMedia/RAGTruth/main/dataset/response.jsonl
!curl -sfL -o data/raw/source_info.jsonl https://raw.githubusercontent.com/ParticleMedia/RAGTruth/main/dataset/source_info.jsonl
!python data/prepare.py > /dev/null && cat data/processed/stats.json | head -20

In [ ]:
!nvidia-smi -L

In [ ]:
# smoke run first: 200 sentences, should take well under a minute
!PYTHONPATH=src python -m hd.baselines --splits val --limit 200 --out /tmp/smoke

In [ ]:
!PYTHONPATH=src python -m hd.baselines --splits val test

In [ ]:
!python eval/evaluate.py --method nli

In [ ]:
!cd /kaggle/working && zip -qr results.zip results && ls -la results